In [33]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

PARQUET_FILE = '../data/clean/nutriscope_ml_sample.parquet'

dataframe = pd.read_parquet(PARQUET_FILE)

display(dataframe.shape)

display(dataframe.dtypes)

display(dataframe.head())

(30041, 21)

code                                  str
product_name                          str
brands                                str
pnns_groups_1                         str
pnns_groups_2                         str
nova_group                          Int64
additives_n                         Int64
energy-kcal_100g                  float64
fat_100g                          float64
saturated-fat_100g                float64
carbohydrates_100g                float64
sugars_100g                       float64
fiber_100g                        float64
proteins_100g                     float64
salt_100g                         float64
fruits-vegetables-legumes_100g    float64
nutriscore_score                    Int64
nutriscore_grade                      str
completeness                      float64
last_modified                      string
split                                 str
dtype: object

,code,product_name,brands,pnns_groups_1,pnns_groups_2,nova_group,additives_n,energy-kcal_100g,fat_100g,saturated-fat_100g,...,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,nutriscore_score,nutriscore_grade,completeness,last_modified,split
0,0000112302660,Soupe potiron et kiri,Liebig,Fruits and vegetables,Soups,4,4,38.300000,1.50000,1.000000,...,2.000000,0.800000,0.90000,0.740000,34.312500,3,c,0.6875,2026-03-14,train
1,0000112302705,velouté de tomates et pointe d'origan,Liebig,Fruits and vegetables,Soups,4,0,28.000000,0.50000,0.300000,...,2.400000,1.000000,0.70000,0.670000,72.977007,1,b,0.7625,2026-03-19,test
2,0000112302754,pur soup' mouliné de 5 légumes verts,liebig,Composite foods,One-dish meals,3,0,24.802376,0.16035,0.044483,...,0.738538,0.837462,0.81314,0.035146,23.242188,0,a,0.9750,2026-03-26,train
3,0000112307261,Velouté de 10 légumes,Liebig,Fruits and vegetables,Soups,4,0,43.000000,1.60000,0.600000,...,1.300000,0.800000,0.70000,0.670000,34.981336,3,c,0.9000,2026-03-26,train
4,0000300000004,Lentilles Brunes Bio,Kazidomi,Composite foods,One-dish meals,3,0,54.000000,0.30000,0.000000,...,0.000000,4.000000,4.00000,0.080000,100.000000,-7,a,0.8875,2026-03-17,test


#### Définitions de fonctions, variables, constantes

In [34]:
SCORING = ["f1_macro", "accuracy", "balanced_accuracy"]
CROSS_VALIDATION = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(y_true, y_pred, nom: str) -> dict[str, float]:
    """Affiche et retourne les métriques principales d'un modèle de classification."""
    metriques = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }

    print(f"=== {nom} ===")
    for cle, valeur in metriques.items():
        print(f"{cle:<12}: {valeur:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))

    return metriques

def evaluate_cross_validation(pipeline: Pipeline, X, y, nom: str) -> dict[str, float | str]:
    '''Evalue un pipeline en validation croisée (moyenne et écart-type par métrique)'''
    scores = cross_validate(
        pipeline, 
        X, 
        y, 
        cv=CROSS_VALIDATION, 
        scoring=SCORING,
        n_jobs=-1
    )

    resume: dict[str, float | str] = {'modele': nom}

    for metric in SCORING:
        values = scores[f'test_{metric}']
        resume[f'{metric}_moy'] = values.mean()
        resume[f'{metric}_std'] = values.std()

    return resume    

def build_pipeline(classifier) -> Pipeline:
    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ]
    )

models = {
    'dummy_most_frequent': DummyClassifier(strategy='most_frequent'),
    'dummy_stratified': DummyClassifier(strategy="stratified", random_state=42),
    'tree_depth_3': DecisionTreeClassifier(max_depth=3, random_state=42),
    'tree_depth_8': DecisionTreeClassifier(max_depth=8, random_state=42),
}

#### Preprocessing

In [35]:
features_to_keep = [
    'energy-kcal_100g',
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'proteins_100g',
    'salt_100g',
    'fiber_100g',
    'fruits-vegetables-legumes_100g',
    'nova_group',
    'additives_n',
    'pnns_groups_1',
    'pnns_groups_2'
]

target = 'nutriscore_grade'

X = dataframe[features_to_keep].copy()
y = dataframe[target].copy()

# Identification des types de colonnes
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.to_list()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.to_list()

print(f"Variables numériques : {numeric_features}")
print(f"Variables catégorielles : {categorical_features}")

Variables numériques : ['energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'proteins_100g', 'salt_100g', 'fiber_100g', 'fruits-vegetables-legumes_100g', 'nova_group', 'additives_n']
Variables catégorielles : ['pnns_groups_1', 'pnns_groups_2']


#### Création des pipelines et des transformers

In [36]:
# Pipeline pour les variables numériques
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # Remplace NaN par la médiane
    ("scaler", StandardScaler())
])

# Pipeline pour les variables catégorielles
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")), # remplace NaN par missing
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Appliquer les pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

#### Séparation train + test

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 20% pour les tests, 80% pour l'entrainement
    random_state=42,
    stratify=y # Maintient les proportions de classes
)

In [38]:
results = [
    evaluate_cross_validation(build_pipeline(classifier), X_train, y_train, name)
    for name, classifier in models.items()
]

comparatif = pd.DataFrame(results).set_index("modele")
comparatif.round(4)

,f1_macro_moy,f1_macro_std,accuracy_moy,accuracy_std,balanced_accuracy_moy,balanced_accuracy_std
modele,,,,,,
dummy_most_frequent,0.0891,0.0000,0.2864,0.0001,0.2000,0.0000
dummy_stratified,0.1987,0.0048,0.2240,0.0052,0.1987,0.0048
tree_depth_3,0.3979,0.0199,0.5368,0.0101,0.4365,0.0114
tree_depth_8,0.7449,0.0074,0.7758,0.0045,0.7430,0.0082


#### Test de prédiction - Arbre de profondeur 3

In [39]:
from sklearn.metrics import confusion_matrix

tree_3 = build_pipeline(models["tree_depth_3"])
tree_3.fit(X_train, y_train)
y_pred_3 = tree_3.predict(X_test)

print(pd.Series(y_pred_3).value_counts())
print(confusion_matrix(y_test, y_pred_3, labels=["a", "b", "c", "d", "e"]))

c    2532
d    1945
e    1420
b     112
Name: count, dtype: int64
[[   0    0  782   12   12]
 [   0   78  518   38   12]
 [   0   24  862  324   37]
 [   0    1  259 1101  228]
 [   0    9  111  470 1131]]


#### Test de prédiction - Arbre de profondeur 8

In [40]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    build_pipeline(models["tree_depth_8"]),
    X_train, y_train,
    cv=CROSS_VALIDATION,
    scoring="f1_macro",
    return_train_score=True,
    n_jobs=-1,
)

print(f"Train : {scores['train_score'].mean():.4f}")
print(f"Val   : {scores['test_score'].mean():.4f}")

Train : 0.7660
Val   : 0.7449
